# Visualization of new tree

In [1]:
import os
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
import re
import math
import numpy as np

In [3]:
# Base folder path
BASE_PATH = 'images'
VALID_SECTIONS = ['dataset', 'trees', 'decision', 'prob', 'calibration', 'scatter', 'true']

# Get list of dataset folders in BASE_PATH
def get_datasets():
    return sorted(os.listdir(BASE_PATH))

# Get list of models (subfolders) inside a given dataset folder, including "root"
def get_models(dataset):
    dataset_path = os.path.join(BASE_PATH, dataset)
    folders = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])
    return ['root'] + folders

# Get image filenames in selected dataset/model that start with the selected section
def get_images(dataset, model, section):
    if model == 'root':
        path = os.path.join(BASE_PATH, dataset)
    else:
        path = os.path.join(BASE_PATH, dataset, model)
    if not os.path.exists(path):
        return []
    return sorted([
        f for f in os.listdir(path)
        if f.endswith('.png') and re.match(rf'^{re.escape(section)}.*\.png$', f)
    ])

# Display images in a 2x2 grid layout, larger and better spaced
def show_images(dataset, model, section):
    output.clear_output()
    with output:
        files = get_images(dataset, model, section)
        if not files:
            print("Image not available")
            return

        path = os.path.join(BASE_PATH, dataset) if model == 'root' else os.path.join(BASE_PATH, dataset, model)

        n = len(files)
        cols = 2
        rows = math.ceil(n / cols)

        # Increase size per image: 8x8 inches per slot
        fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 7 * rows), gridspec_kw={'hspace': 0.3})

        if rows == 1 and cols == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = np.expand_dims(axes, axis=0)
        elif cols == 1:
            axes = np.expand_dims(axes, axis=1)

        axes = axes.flatten()

        for ax, fname in zip(axes, files):
            img = Image.open(os.path.join(path, fname))
            ax.imshow(img)
            ax.set_title(fname, fontsize=14)
            ax.axis('off')

        for ax in axes[len(files):]:
            ax.axis('off')

        #plt.tight_layout()
        plt.show()

# === Dropdown widgets ===
dataset_dropdown = widgets.Dropdown(
    options=get_datasets(), description='Dataset:', layout=widgets.Layout(width='200px'))

model_dropdown = widgets.Dropdown(
    description='Model:', layout=widgets.Layout(width='200px'))

section_dropdown = widgets.Dropdown(
    options=VALID_SECTIONS, description='Section:', layout=widgets.Layout(width='200px'))

# === Output before callbacks ===
output = widgets.Output()

# === Callback functions ===
def update_models(*args):
    model_dropdown.options = get_models(dataset_dropdown.value)
    model_dropdown.value = model_dropdown.options[0] if model_dropdown.options else None

def update_images(*args):
    if dataset_dropdown.value and model_dropdown.value and section_dropdown.value:
        show_images(dataset_dropdown.value, model_dropdown.value, section_dropdown.value)

# === Link widget changes to update functions ===
dataset_dropdown.observe(update_models, names='value')
dataset_dropdown.observe(update_images, names='value')
model_dropdown.observe(update_images, names='value')
section_dropdown.observe(update_images, names='value')

# === Initialize interface ===
update_models()

# Layout and display
controls = widgets.HBox([dataset_dropdown, model_dropdown, section_dropdown])
display(widgets.VBox([controls, output]))
update_images()
